# One-Proportion Z-Test (Post-Test Analysis)

**Method:** Frequentist z-test for a single proportion — tests whether observed conversion rate differs from a target benchmark.

---

$$z = \frac{\hat{p} - p_0}{\sqrt{\frac{p_0(1 - p_0)}{n}}}$$

---

| Symbol | Meaning |
|--------|---------|
| $z$ | Test statistic (standard normal scale) |
| $\hat{p}$ | Observed conversion rate (= conversions / visitors) |
| $p_0$ | Target benchmark conversion rate (null hypothesis) |
| $n$ | Number of visitors (sample size) |

### Wilson Score Confidence Interval

$$\tilde{p} = \frac{\hat{p} + \frac{z_{\alpha/2}^2}{2n} \pm z_{\alpha/2}\sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z_{\alpha/2}^2}{4n^2}}}{1 + \frac{z_{\alpha/2}^2}{n}}$$

### Hypothesis table

| Test Type | Null Hypothesis $H_0$ | Alternative $H_1$ | Use when... |
|-----------|----------------------|-------------------|-------------|
| **One-sided** (片側検定) | $p = p_0$ | $p > p_0$ | You want to prove CVR is **above** target |
| **Two-sided** (両側検定) | $p = p_0$ | $p \neq p_0$ | CVR could be **above or below** target |

## Data Input

Enter your observed data and test configuration below.

In [34]:
import json

config = json.load(open("../test_config.json"))
d = config["design"]
planned_n = config["computed"]["planned_n"]

# ==========================================
#  OBSERVED DATA — enter your results here
# ==========================================
visitors    = 700    # unique visitors
conversions = 7      # unique conversions
# ==========================================

In [35]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

def run_frequentist_post_test(visitors, conversions, target_rate, two_sided=False, alpha=0.05, n_variants=1):
    alpha_adj = alpha / n_variants
    p_hat = conversions / visitors
    se = np.sqrt(target_rate * (1 - target_rate) / visitors)
    z_stat = (p_hat - target_rate) / se
    z_crit = norm.ppf(1 - alpha_adj / 2) if two_sided else norm.ppf(1 - alpha_adj)
    test_label = "Two-Sided" if two_sided else "One-Sided"

    p_value = 2 * (1 - norm.cdf(abs(z_stat))) if two_sided else 1 - norm.cdf(z_stat)

    # Wilson score CI
    z_ci = norm.ppf(1 - alpha_adj / 2)
    denom = 1 + z_ci**2 / visitors
    center = (p_hat + z_ci**2 / (2 * visitors)) / denom
    margin = (z_ci * np.sqrt(p_hat * (1 - p_hat) / visitors + z_ci**2 / (4 * visitors**2))) / denom
    ci_lower, ci_upper = center - margin, center + margin

    reject = p_value < alpha_adj

    # --- Output ---
    print(f"--- POST-TEST Z-TEST ({test_label}) ---")
    print(f"手法 : 頻度論 z検定（単一比率の検定）")
    print(f"KPI  : コンバージョン率（CVR）")
    print(f"目標 : {target_rate:.1%}")
    if n_variants > 1:
        print(f"補正 : Bonferroni（{n_variants}バリアント） α_adj = {alpha_adj}")
    print(f"-" * 50)
    print(f"【観測データ】")
    print(f"  訪問者数 (UU)  : {visitors:,}")
    print(f"  CV数 (UU)      : {conversions:,}")
    print(f"  実測CVR        : {p_hat:.4f} ({p_hat:.2%})")
    print(f"-" * 50)
    print(f"α = {alpha_adj}  →  z_crit = {z_crit:.4f}")
    print(f"z = (p̂ - p₀) / SE = ({p_hat:.4f} - {target_rate}) / {se:.6f} = {z_stat:.4f}")
    print(f"p値 = {p_value:.6f}")
    print(f"Wilson {1-alpha_adj:.0%} CI: [{ci_lower:.2%}, {ci_upper:.2%}]")
    print(f"-" * 50)

    if reject:
        verdict_en = "REJECT H₀"
        print(f"判定 : p値 ({p_value:.6f}) < α ({alpha_adj}) → 帰無仮説を棄却")
        if two_sided:
            print(f"解釈 : 観測CVR {p_hat:.2%} は目標 {target_rate:.1%} と統計的に有意な差がある")
        else:
            print(f"解釈 : 観測CVR {p_hat:.2%} は目標 {target_rate:.1%} を統計的に有意に上回っている")
    else:
        verdict_en = "FAIL TO REJECT H₀"
        print(f"判定 : p値 ({p_value:.6f}) ≥ α ({alpha_adj}) → 帰無仮説を棄却できず")
        if two_sided:
            print(f"解釈 : 観測CVR {p_hat:.2%} と目標 {target_rate:.1%} の差は統計的に有意とは言えない")
        else:
            print(f"解釈 : 観測CVR {p_hat:.2%} が目標 {target_rate:.1%} を上回るとは言えない")
    print(f"-" * 50)

    # --- Plot ---
    x_range = max(4, abs(z_stat) + 1)
    x = np.linspace(-x_range, x_range, 1000)
    y = norm.pdf(x, 0, 1)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name='N(0,1)', line=dict(color='#1f77b4', width=3)))

    # Rejection region(s)
    if two_sided:
        for sign, label in [(1, f'z ≥ {z_crit:.4f}'), (-1, f'z ≤ -{z_crit:.4f}')]:
            mask = (x * sign) >= z_crit
            fig.add_trace(go.Scatter(x=x[mask], y=y[mask], mode='lines', fill='tozeroy',
                name=f'Rejection ({label})', line=dict(width=0), fillcolor='rgba(214,39,40,0.3)'))
        fig.add_vline(x=z_crit, line_dash="dash", line_color="gray", annotation_text=f"z_crit = {z_crit:.4f}", annotation_position="top right")
        fig.add_vline(x=-z_crit, line_dash="dash", line_color="gray", annotation_text=f"-z_crit = -{z_crit:.4f}", annotation_position="top left")
    else:
        mask = x >= z_crit
        fig.add_trace(go.Scatter(x=x[mask], y=y[mask], mode='lines', fill='tozeroy',
            name=f'Rejection (z ≥ {z_crit:.4f})', line=dict(width=0), fillcolor='rgba(214,39,40,0.3)'))
        fig.add_vline(x=z_crit, line_dash="dash", line_color="gray", annotation_text=f"z_crit = {z_crit:.4f}", annotation_position="top right")

    fig.add_trace(go.Scatter(x=[z_stat], y=[norm.pdf(z_stat, 0, 1)], mode='markers',
        name=f'Observed z = {z_stat:.4f}', marker=dict(color='red', size=12, symbol='diamond')))
    fig.add_vline(x=z_stat, line_dash="dot", line_color="red", annotation_text=f"z = {z_stat:.4f}", annotation_position="bottom right")

    bonf = f"  |  Bonferroni: α_adj = {alpha_adj}" if n_variants > 1 else ""
    fig.update_layout(
        title=dict(text=f"<b>Frequentist Post-Test Z-Test ({test_label})</b><br>"
            f"H₀: p = {target_rate:.1%} | z = {z_stat:.4f} | p = {p_value:.4f} → <b>{verdict_en}</b>{bonf}", font=dict(size=20)),
        xaxis_title="z", yaxis_title="Density",
        xaxis=dict(showgrid=True, gridcolor='#eee'), yaxis=dict(showgrid=True, gridcolor='#eee'),
        template="plotly_white", plot_bgcolor='rgba(0,0,0,0)')
    fig.show()

In [36]:
def sequential_check_frequentist(visitors, conversions, target_rate, planned_n, two_sided=True, alpha=0.05, n_variants=1):
    """O'Brien-Fleming alpha spending for early stopping in sequential testing."""
    alpha_adj = alpha / n_variants
    t = visitors / planned_n
    p_hat = conversions / visitors
    se = np.sqrt(target_rate * (1 - target_rate) / visitors)
    z_stat = (p_hat - target_rate) / se

    z_full = norm.ppf(1 - alpha_adj / 2) if two_sided else norm.ppf(1 - alpha_adj)
    z_boundary = z_full / np.sqrt(t)

    stop_early = abs(z_stat) >= z_boundary if two_sided else z_stat >= z_boundary

    print(f"--- SEQUENTIAL CHECK (O'Brien-Fleming) ---")
    if n_variants > 1:
        print(f"補正 : Bonferroni（{n_variants}バリアント） α_adj = {alpha_adj}")
    print(f"進捗 : {visitors:,} / {planned_n:,} ({t*100:.1f}%)")
    print(f"z統計量 = {z_stat:.4f}  |  OBF境界値 = {z_boundary:.4f}")
    print(f"-" * 50)

    if stop_early:
        print(f"判定 : STOP EARLY — 統計的に有意な差を検出。計画サンプル数を待たず結論可能")
    elif t >= 1.0:
        print(f"判定 : COMPLETE — 計画サンプル数到達。通常のz検定で最終判定してください")
    else:
        remaining = planned_n - visitors
        print(f"判定 : CONTINUE — 早期停止基準未達。残り約 {remaining:,}人")
    print(f"-" * 50)

In [37]:
run_frequentist_post_test(visitors=visitors, conversions=conversions,
    target_rate=d["target_rate"], two_sided=d["two_sided"],
    alpha=d["alpha"], n_variants=d["n_variants"])

if planned_n:
    print()
    sequential_check_frequentist(visitors=visitors, conversions=conversions,
        target_rate=d["target_rate"], planned_n=planned_n,
        two_sided=d["two_sided"], alpha=d["alpha"], n_variants=d["n_variants"])

--- POST-TEST Z-TEST (Two-Sided) ---
手法 : 頻度論 z検定（単一比率の検定）
KPI  : コンバージョン率（CVR）
目標 : 2.0%
--------------------------------------------------
【観測データ】
  訪問者数 (UU)  : 700
  CV数 (UU)      : 7
  実測CVR        : 0.0100 (1.00%)
--------------------------------------------------
α = 0.05  →  z_crit = 1.9600
z = (p̂ - p₀) / SE = (0.0100 - 0.02) / 0.005292 = -1.8898
p値 = 0.058782
Wilson 95% CI: [0.49%, 2.05%]
--------------------------------------------------
判定 : p値 (0.058782) ≥ α (0.05) → 帰無仮説を棄却できず
解釈 : 観測CVR 1.00% と目標 2.0% の差は統計的に有意とは言えない
--------------------------------------------------



--- SEQUENTIAL CHECK (O'Brien-Fleming) ---
進捗 : 700 / 1,283 (54.6%)
z統計量 = -1.8898  |  OBF境界値 = 2.6535
--------------------------------------------------
判定 : CONTINUE — 早期停止基準未達。残り約 583人
--------------------------------------------------
